
# Dashboard Context and Goals

The Alpine Swift (*Tachymarptis melba*) is a small areal insectivore that spends most of their life in air. They are capable of migrating to their wintering ground in africa in approximately one week, which is among the shortest migration times of any long-distance migrant [@meier2020]. In a long term study between 2014 and 2016 the Swiss Ornithological Insitute has trached their trans-Saharan migratory cycle with the use of light-level geolocators. For this they tagged Alpine Swift Individuals of four populations along a twelve-degree latitudinal gradient: Pirasali Island (Turkey), Tarragona (Spain), Sofia (Bulgaria) and different locations in Switzerland. The headline finding of their study was that the eastern and western populations follow two geographically distinct flyways across the Sahara. The western populations migrate over the Iberian peninsula, while the eastern populations migrate over egypt. Further, more southern colonies depart for their wintering grounds later in autumn, while more northern colonies depart from their wintering grounds later in spring, resulting in a breeding season that is up to 48 days shorter for the most northern population compared to the most southern one [@meier2020]. The aim of our dashboard is to provide an analysis tool that resolves around analysis this data.

---

# Data

The dashboard is based on data collected during this long-term research project. The dataset comprises 215 tagged Alpine Swift individuals from nine colonies: six from Switzerland, and one each from Turkey, Spain, and Bulgaria. Individual locations were determined using light-level geolocators, which record light levels twice daily to estimate position from local sunrise and sunset times [@meier2020]. The data is hosted as 9 different datasets on movebank, each dataset holding the entire 3 year study data for one colony [@meier2020data]. The combined dataset holds a total of 268'158 datapoints. It is important to point out that datapoints captured during summer are not provided, as it is not possible to rely on the geolocator readings, as repeatably entering and exiting the nest cavities introduces shading which is then interpreted as a sunrise or sunset event which then reports unreasonable location data. 

---

# Story telling concept and intended insights

The aim of geovisual analytics is to detect the expected while being able to discover the unexpected (Jim Thomas). To provide this, the geovisual analytics approach follows schneidermanns information seeking mantra which focuses on first providing an overwiev over the whole data, then providing utilities to zoom and filter data and lastly optaining additional details on demand (Schneiderman ). The story telling concept follows this mantra, first providing an overwiev over the data, then providing tools to zoom and filter data and then lastly providing details on demand to anable the testing of hypothesises. This then translates to our story. First the user should uncover some obvious facts:

- there is migration between europe and africa happening. 
- there are two main routes of travel. The iberian peninsula and egypt.


Then Filtering and zomming actions should enable the user to discover, that higher latitude colonies depart earlier from europe and depart later from africa. Lastly the user should be able to discover other patterns hidden inside the data on its own once the expected is explored.

---

# Key dashboard Functionalities

## Spatial Map

At the center of the dashboard sits a map. which serves a dual purpose both for static data display aswell as animation. If no annimation is running, the display shows a high level overwiev of the datapoints in the dataset. This implements the "Overview first" approach of Schneidermanns mantra of geovisualisation because as the user enters the dashboard, he or she will immediately be meeted with this large static map.

When an animation is beeing run, the map shows the temporal developement of the location of different alpine swift individuals.

## Positional uncertainty layer

## Filter Menue

The filter menue can be used to filter the displayed data after various attributes. Different colonies can be togelled on and of over a drop down menue, a slider can be used to select from which years datapoints should be displayed. Some "quick filters" are allready implemented for the user. For instance the western and eastern flight ways can be filtered for. This should help the reader to follow our intended storytelling and arrive at the insight, that there are two main flight paths between europe and africa.


## Geovisual anayltics interactions

The dashboard also implements the main geovisual analytics interactions through its functionalities

- **Panning, zooming, navigation:** This is implemented over the spatial map at the center of the dashboard

- **Attribute re-expression:** This interaction is provided below the filter menue. Datapoints can be collored according to their group memberships along different variables.Example: A datapoint can be collored according to their colony membership (who which colony does the individual belong) or according to its year membership (from which year the datapoint is from). This allows the user to view the data from many different dimensions.

- **Brushing/Mouse over**: Users can hower over specific datapoints in the map to recieve additional infromation about the datapoint (individual tag, lat. lon., colony membership, ...)

- **Linked views**: Two plots, linked to the spatial map are provided. A set of boxplots displays the current latitudual distribution of individuals along a selected attribute (e.g. colony membership, flyway). This boxplot also develops temporally if an animation is played. The second linked plot shows a line plot of the latitudual position of every individual over time. Additionaly an attribute of choice is used to color the lines according to group memberships (e.g. colony membership, flyway). The lineplot features temporal animation too, however also assumes an important purpose as a static plot. it easy to the user to detect the fact, that higher latitudual colonies departure earlier to europe and departure later to africa. Therefore this plot is central in implementing the intended narrative and insight.

- **Animation**: Like previously mentioned, the dashboard also features temporal animation.


---

# Code For the dashboard

In [2]:
source("R/data_acquisition.R")
source("R/data_processing.R")

In [3]:
processed <- build_processed_data()

Using cached processed data: data/processed/tracks_processed.rds


## Helpers

In [4]:
# =============================================================================
# R/helpers.R - shared utilities (palettes, formatters, cartographic config)
# -----------------------------------------------------------------------------
# Cartographic principles applied (after Slocum et al., Thematic Cartography
# and Geovisualization, 3rd ed.):
#   * Visual hierarchy: subdued grey basemap, thematic symbols dominate.
#   * Sequential data:  perceptually uniform ramps (viridis / cividis),
#                       NEVER rainbow/spectral.
#   * Qualitative data: ColorBrewer-style palette, restricted to <= 8 hues,
#                       colorblind-tested (Okabe-Ito derived).
#   * Diverging data:   reserved for centered measurements (not used).
#   * Symbol scaling:   area-proportional (sqrt(n)) for quantities.
# =============================================================================

suppressPackageStartupMessages({
  library(RColorBrewer)
  library(viridisLite)
  library(scales)
})

# ---- Qualitative palette: 4 populations (Okabe-Ito-derived, CB-safe) -------
# Switzerland = teal-green (subdued, central population)
# Spain       = warm orange (western, southern, distinct from CH)
# Bulgaria    = deep purple (eastern, contrasts with CH/ES)
# Turkey      = burnt sienna (eastern, distinct from BG, less saturated than
#               the previous magenta which was too aggressive for science viz)
# ColorBrewer "Dark2" - the canonical qualitative palette Slocum (ch.14)
# recommends for nominal categorical data. Colour-blind-tested, print-safe.
COUNTRY_COLORS <- c(
  "Switzerland" = "#1B9E77",   # Dark2-1 teal
  "Spain"       = "#D95F02",   # Dark2-2 orange
  "Bulgaria"    = "#7570B3",   # Dark2-3 purple
  "Turkey"      = "#E7298A"    # Dark2-4 magenta
)

# Flyway: same Dark2 family as the country palette so the colour scheme
# is consistent whether the user groups by country or by flyway.
FLYWAY_COLORS <- c(
  "western" = "#1B9E77",   # Dark2-1 teal
  "eastern" = "#D95F02"    # Dark2-2 orange
)

# Year: ColorBrewer "YlGnBu" 3-step (sequential, Slocum ch.14 - ordinal data
# needs a sequential ramp, not a qualitative one).
YEAR_COLORS <- c(
  "2014" = "#EDF8B1",
  "2015" = "#7FCDBB",
  "2016" = "#2C7FB8"
)

# Phase colors (nominal, but consistent with annual-cycle order)
PHASE_COLORS <- c(
  "breeding"   = "#1B7837",   # forest green
  "migration"  = "#E08214",   # amber (transit)
  "wintering"  = "#542788"    # deep purple (residence in tropics)
)

# Sequential ramp for time / count (perceptually uniform, colorblind-safe)
SEQ_RAMP <- function(n = 9, option = "viridis") {
  viridisLite::viridis(n, option = option, direction = 1, end = 0.95)
}

# Subdued base map (CartoDB Positron) - pale grey, low chroma. The pale
# basemap is intentional: figure / ground principle (Slocum ch. 12) means
# the thematic symbols (bird tracks) must dominate, not the substrate.
BASEMAP_URL <- "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png"
BASEMAP_ATTR <- paste0(
  "&copy; <a href=\"https://www.openstreetmap.org/copyright\">OpenStreetMap</a> ",
  "contributors &copy; <a href=\"https://carto.com/attributions\">CARTO</a>"
)

# Map extent: covers all breeding colonies + entire trans-Saharan migration
# corridor down to the wintering grounds at 5 N.
MAP_BOUNDS <- list(
  lng1 = -18, lat1 = -5,
  lng2 =  45, lat2 = 55
)

# Day-of-year helpers
doy_to_md <- function(doy) {
  format(as.Date(doy - 1, origin = "2015-01-01"), "%b %d")
}
doy_to_long <- function(doy) {
  format(as.Date(doy - 1, origin = "2015-01-01"), "%d %B")
}

stage_label <- function(phase) {
  tools::toTitleCase(phase)
}

# Quick formatter for big integers
fmt_int <- function(x) format(x, big.mark = " ", scientific = FALSE)

# ---- Shared "color by" palette resolver ------------------------------------
# Used by both the static map and the animated map so the colour scheme is
# consistent and the legend always matches what is drawn.
#
# Returns a list:
#   levels  -> character vector of category labels (sorted)
#   colors  -> named character vector keyed by levels (the palette)
#   cat     -> character vector aligned with df rows (the category each row
#              belongs to under the current grouping)
#   col     -> character vector aligned with df rows (the colour of each row)
#   title   -> short legend title for the grouping
resolve_palette <- function(df, gm) {
  if (nrow(df) == 0) {
    return(list(levels = character(0), colors = character(0),
                cat = character(0), col = character(0),
                title = "Population"))
  }
  levels <- switch(gm,
                   country = sort(unique(df$country)),
                   colony  = sort(unique(df$colony_name)),
                   flyway  = c("western", "eastern"),
                   year    = sort(unique(as.character(df$year))))
  colors <- switch(
    gm,
    country = COUNTRY_COLORS[levels],
    colony  = setNames(
      grDevices::colorRampPalette(
        suppressWarnings(RColorBrewer::brewer.pal(8, "Dark2"))
      )(length(levels)),
      levels),
    flyway  = FLYWAY_COLORS[levels],
    year    = YEAR_COLORS[levels])
  # Defensive: any unknown category gets a neutral grey
  colors[is.na(colors)] <- "#999999"

  cat_vec <- switch(gm,
                    country = df$country,
                    colony  = df$colony_name,
                    flyway  = df$flyway,
                    year    = as.character(df$year))
  # IMPORTANT: unname the per-row colours. If we leave the names on,
  # jsonlite serialises the vector as a JSON OBJECT (collapsing duplicate
  # category names to a single key) and every Leaflet marker ends up
  # with the same colour. unname() forces an unnamed vector -> JSON array.
  col_vec <- unname(colors[cat_vec])
  col_vec[is.na(col_vec)] <- "#999999"

  list(levels = levels, colors = colors,
       cat = unname(cat_vec), col = col_vec,
       title = switch(gm,
                      country = "Population",
                      colony  = "Colony",
                      flyway  = "Flyway",
                      year    = "Year"))
}

# Render the palette as an HTML legend (shared between map modules).
render_map_legend <- function(pal, extra_row = NULL) {
  if (length(pal$levels) == 0) {
    return(htmltools::HTML(
      '<div class="map-legend"><div class="legend-title">No data</div></div>'))
  }
  rows <- mapply(function(lvl, col) {
    sprintf('<div class="legend-row"><span class="legend-swatch" style="background:%s"></span>%s</div>',
            col, htmltools::htmlEscape(lvl))
  }, pal$levels, pal$colors, USE.NAMES = FALSE)
  htmltools::HTML(sprintf(
    '<div class="map-legend">
       <div class="legend-title">%s</div>
       %s
       %s
     </div>',
    pal$title,
    paste(rows, collapse = "\n"),
    if (is.null(extra_row)) "" else
      paste0("<hr/>", extra_row)
  ))
}


## mod_animation

In [5]:
# =============================================================================
# R/mod_animation.R - Merged map (static + server-side animation)
# -----------------------------------------------------------------------------
# Implements the Meier et al. 2020 Figure 1 visualisation language:
#
#   * Path-resample dots (low alpha): every retained fix is drawn at once.
#     We never interpolate fake positions between sparse / jittery samples;
#     we instead overlay many samples to convey the credible path.
#   * Migration phase: vertical "shadow strokes" running from the per-day
#     lat_lo to lat_hi credible band visualise the well-known latitudinal
#     uncertainty of light-level geolocators, instead of a hard line.
#   * Moving-point trail (sperm trail): optional polyline trailing each
#     active individual through the last N days of the cycle.
#   * Current day: slightly translucent bright markers, drawn on its own
#     pane so it is ALWAYS on top of every other layer.
#   * Colour is decoupled from data: changing "Color by" only repaints
#     existing dots and updates the on-map legend.
#   * Clicking a point selects the bird and draws its full bird-year track;
#     clicking the same bird again deselects.
# =============================================================================
suppressPackageStartupMessages({
  library(shiny)
  library(bslib)
  library(leaflet)
  library(dplyr)
  library(ggplot2)
  library(plotly)
  library(sf)
  library(htmltools)
})

animation_ui <- function(id) {
  ns <- NS(id)
  tagList(
    bslib::layout_columns(
      col_widths = c(8, 4),
      card(
        full_screen = TRUE,
        leafletOutput(ns("anim_map"), height = "65vh"),
        div(class = "plot-caption",
            "Trans-Saharan annual cycle of Alpine Swifts. ",
            "Each dot is one retained daily fix (alpha-blended path ",
            "resamples); brighter circles mark the slider's current ",
            "day. Vertical strokes show the per-bird 10-90 percentile ",
            "latitudinal uncertainty band during migration. ",
            "Toggle layers top-right; click a dot to highlight that ",
            "bird; click again to deselect.")
      ),
      card(
        card_header(span("Playback Controls",
                         class = "panel-heading fw-bold")),
        card_body(
          div(class = "d-flex justify-content-between mb-3",
              actionButton(ns("playBtn"), label = "Play",
                           class = "btn-success flex-grow-1 me-2"),
              actionButton(ns("resetBtn"), label = "Reset",
                           class = "btn-outline-secondary flex-grow-1 ms-2")
          ),
          div(class = "text-center fs-5 fw-bold text-success mb-2",
              textOutput(ns("date_label"), inline = TRUE)),
          sliderInput(ns("doy"), label = NULL,
                      min = 1, max = 365, value = 100, step = 1,
                      width = "100%"),
          hr(),
          radioButtons(ns("speed_choice"), "Speed",
                       choices  = c("Slow" = "slow", "Medium" = "med",
                                    "Fast" = "fast"),
                       selected = "med", inline = TRUE),
          radioButtons(ns("trail_choice"), "Moving-point trail",
                       choices  = c("Off" = 0, "1 day" = 1,
                                    "3 days" = 3, "7 days" = 7),
                       selected = 0, inline = TRUE),
          hr(),
          h6("Selected individual", class = "panel-subheading"),
          uiOutput(ns("bird_card"))
        )
      )
    ),
    fluidRow(
      column(12,
             card(card_header("Latitude of active individuals"),
                  plotlyOutput(ns("anim_lat_curve"), height = "260px"),
                  div(class = "plot-caption",
                      "Boxplot of latitude (degrees N) for every bird ",
                      "active on the slider's day-of-year, grouped by ",
                      "the active 'Color by' category. Whiskers extend ",
                      "to 1.5x IQR; jitter points show individual fixes."))
      )
    )
  )
}

animation_server <- function(id, filtered, processed) {
  moduleServer(id, function(input, output, session) {
    ns <- session$ns

    is_playing <- reactiveVal(FALSE)
    sel_bird   <- reactiveVal(NULL)

    observeEvent(input$playBtn, { is_playing(!is_playing()) })
    observeEvent(input$resetBtn, {
      is_playing(FALSE)
      sel_bird(NULL)
      updateSliderInput(session, "doy", value = 100)
      # Reset the layer flip: show Path resamples, hide Current day, and
      # rearm anim_started so the next Play press flips again.
      leafletProxy(ns("anim_map")) %>%
        showGroup("Path resamples") %>%
        hideGroup("Current day")
      anim_started(FALSE)
    })

    observe({
      if (!is_playing()) return()
      # Slower base tick (300 ms) so the user has time to react to Pause and
      # the displayed current-day marker stays in sync with the slider.
      invalidateLater(300)
      isolate({
        cur  <- input$doy %||% 100
        step <- switch(input$speed_choice %||% "med",
                       slow = 1, med = 2, fast = 4)
        new_val <- if (cur + step > 365) 1 else cur + step
        updateSliderInput(session, "doy", value = new_val)
      })
    })

    plot_doy <- reactive({ as.numeric(input$doy %||% 100) }) |>
      shiny::throttle(350)

    # ---- Current-frame slice (used for highlight + latitude plot) ----------
    # Tight +/- 1 day window: we want the visible current-day dot to actually
    # match the slider's day. With daily aggregation there is at most one row
    # per (bird, date), so the slice picks the bird's nearest fix within that
    # 3-day window centred on d_f. Birds with no fix in the window simply do
    # not appear for that frame - which is the honest representation.
    current_frame_data <- reactive({
      df <- filtered$daily()
      if (is.null(df) || nrow(df) == 0) return(NULL)
      d_f <- plot_doy()
      cur <- df %>%
        dplyr::filter(abs(doy - d_f) <= 1) %>%
        dplyr::group_by(bird_id, year) %>%
        dplyr::slice(which.min(abs(doy - d_f))) %>%
        dplyr::ungroup()
      if (nrow(cur) == 0) return(NULL)
      gm <- filtered$group_mode()
      cur$group_col <- switch(gm,
                              country = cur$country,
                              colony  = cur$colony_name,
                              flyway  = cur$flyway,
                              year    = as.character(cur$year))
      attr(cur, "palette_colors") <- resolve_palette(cur, gm)$colors
      cur
    })

    output$date_label <- renderText({
      format(as.Date(as.integer(input$doy %||% 100) - 1,
                     origin = "2015-01-01"), "%d %B")
    })

    # ---- Initial leaflet skeleton (with explicit z-order panes) ------------
    output$anim_map <- renderLeaflet({
      leaflet(options = leafletOptions(worldCopyJump = FALSE,
                                       minZoom = 2, maxZoom = 9,
                                       preferCanvas = TRUE)) |>
        addTiles(urlTemplate = BASEMAP_URL,
                 attribution = BASEMAP_ATTR,
                 options = tileOptions(opacity = 0.9)) |>
        fitBounds(lng1 = MAP_BOUNDS$lng1, lat1 = MAP_BOUNDS$lat1,
                  lng2 = MAP_BOUNDS$lng2, lat2 = MAP_BOUNDS$lat2) |>
        # Panes: higher zIndex = drawn on top. Current day = top.
        addMapPane("paneShadow",    zIndex = 380) |>
        addMapPane("paneTrail",     zIndex = 410) |>
        addMapPane("paneResamples", zIndex = 425) |>
        addMapPane("paneSelected",  zIndex = 440) |>
        addMapPane("paneCurrent",   zIndex = 470) |>
        addLayersControl(
          overlayGroups = c("Path resamples",
                            "Migration uncertainty",
                            "Current day"),
          options = layersControlOptions(collapsed = FALSE,
                                         autoZIndex = FALSE)) |>
        # Initial visibility: only Path resamples on. Migration uncertainty
        # is toggled off by default; Current day appears once Play is hit.
        hideGroup("Migration uncertainty") |>
        hideGroup("Current day")
    })

    # ---- Layer flip on Play -----------------------------------------------
    # Every Play press re-activates the Current day layer. The first Play
    # additionally hides Path resamples for good. Pause / Reset never touch
    # either group, so the current-day view persists across pauses.
    anim_started <- reactiveVal(FALSE)
    observeEvent(is_playing(), {
      if (!isTRUE(is_playing())) return()    # only react to play starts
      proxy <- leafletProxy(ns("anim_map"))
      proxy %>% showGroup("Current day")     # always: turn Current day on
      if (!isTRUE(anim_started())) {
        proxy %>% hideGroup("Path resamples")
        anim_started(TRUE)
      }
    }, ignoreInit = TRUE)

    # ---- Map legend (adapts to "Color by") ---------------------------------
    observe({
      df <- filtered$daily()
      gm <- filtered$group_mode()
      proxy <- leafletProxy(ns("anim_map")) %>%
        removeControl("anim_legend")
      if (is.null(df) || nrow(df) == 0) return()
      pal <- resolve_palette(df, gm)
      html <- render_map_legend(pal)
      proxy %>% addControl(html, position = "topright",
                           layerId = "anim_legend")
    })

    # ---- Always-on path resamples (figure-1 dot cloud) ---------------------
    # Repaints when the *data filter* or the *colour mode* changes, but the
    # visible *set of points* is determined only by the data filter -> the
    # "Color by" radio only changes the colours, not the visible dots.
    observe({
      df <- filtered$daily()
      gm <- filtered$group_mode()
      proxy <- leafletProxy(ns("anim_map")) %>%
        clearGroup("Path resamples") %>%
        clearGroup("Migration uncertainty")

      if (is.null(df) || nrow(df) == 0) return()

      pal <- resolve_palette(df, gm)
      df$col <- pal$col

      # Phase-aware alpha: breeding/wintering points are dense -> lower alpha;
      # migration points are sparse -> a bit brighter. All translucent enough
      # that overlapping dots remain visible.
      df$alpha <- ifelse(df$phase == "migration", 0.45, 0.22)

      # Layer 1 - migration uncertainty "shadow strokes".
      mig <- df[df$phase == "migration" &
                  is.finite(df$lat_lo) & is.finite(df$lat_hi) &
                  (df$lat_hi - df$lat_lo) > 0.05, , drop = FALSE]
      if (nrow(mig) > 0) {
        geoms <- lapply(seq_len(nrow(mig)), function(i) {
          sf::st_linestring(rbind(
            c(mig$lon[i], mig$lat_lo[i]),
            c(mig$lon[i], mig$lat_hi[i])))
        })
        unc_sf <- sf::st_sf(col = mig$col,
                            bird_id = mig$bird_id,
                            geometry = sf::st_sfc(geoms, crs = 4326))
        proxy %>% addPolylines(
          data    = unc_sf,
          color   = ~col,
          weight  = 5,
          opacity = 0.10,
          group   = "Migration uncertainty",
          options = pathOptions(interactive = FALSE, pane = "paneShadow"))
      }

      # Layer 2 - path-resample dot cloud (delimiter "##" cannot appear in IDs)
      proxy %>% addCircleMarkers(
        data        = df,
        lng         = ~lon,
        lat         = ~lat,
        layerId     = ~paste("pr", bird_id, year, doy, sep = "##"),
        radius      = ifelse(df$phase == "migration", 3.0, 2.5),
        color       = ~col,
        fillColor   = ~col,
        stroke      = FALSE,
        fillOpacity = df$alpha,
        group       = "Path resamples",
        label       = ~paste0(bird_id, " - ", country,
                              " - ", format(date, "%d %b %Y")),
        options     = pathOptions(pane = "paneResamples"))
    })

    # ---- Moving-point trail (sperm trail) ----------------------------------
    observe({
      proxy <- leafletProxy(ns("anim_map")) %>%
        clearGroup("Moving trail")
      trail_n <- as.integer(input$trail_choice %||% 0)
      if (trail_n <= 0) return()
      df <- filtered$daily()
      gm <- filtered$group_mode()
      if (is.null(df) || nrow(df) == 0) return()

      d_f <- plot_doy()
      # Anchor the trail at each bird's NEAREST fix to d_f using the same
      # +/- 1 day window that current_frame_data() uses, so the polyline
      # ends exactly at the bright current-day marker.
      cur_anchor <- df %>%
        dplyr::filter(abs(doy - d_f) <= 1) %>%
        dplyr::group_by(bird_id, year) %>%
        dplyr::slice(which.min(abs(doy - d_f))) %>%
        dplyr::ungroup() %>%
        dplyr::select(bird_id, year, current_doy = doy)

      if (nrow(cur_anchor) == 0) return()

      paths <- df %>%
        dplyr::inner_join(cur_anchor, by = c("bird_id", "year")) %>%
        dplyr::filter(doy <= current_doy,
                      doy >= (current_doy - trail_n)) %>%
        dplyr::arrange(bird_id, year, doy)

      if (nrow(paths) == 0) return()

      pal <- resolve_palette(df, gm)
      paths$col <- unname(pal$colors[switch(gm,
                                            country = paths$country,
                                            colony  = paths$colony_name,
                                            flyway  = paths$flyway,
                                            year    = as.character(paths$year))])
      paths$col[is.na(paths$col)] <- "#999999"

      # 1. Polylines: connect each bird-year's last trail_n daily fixes.
      groups <- split(paths, paste(paths$bird_id, paths$year, sep = "|"))
      groups_for_line <- groups[vapply(groups, nrow, integer(1)) >= 2L]
      if (length(groups_for_line) > 0) {
        geoms <- lapply(groups_for_line, function(g)
          sf::st_linestring(cbind(as.numeric(g$lon), as.numeric(g$lat))))
        meta  <- do.call(rbind, lapply(groups_for_line,
                                       function(g) g[1, c("bird_id", "col"),
                                                     drop = FALSE]))
        trail_sf <- sf::st_sf(meta,
                              geometry = sf::st_sfc(geoms, crs = 4326))
        proxy %>% addPolylines(
          data    = trail_sf,
          color   = ~col,
          weight  = 2.2,
          opacity = 0.40,
          group   = "Moving trail",
          options = pathOptions(interactive = FALSE, pane = "paneTrail"))
      }

      # 2. Small dots: one per fix in the trail (smaller than the bright
      #    current-day marker, larger than the path-resample dots, so the
      #    trail reads as the bird's recent trajectory).
      proxy %>% addCircleMarkers(
        data        = paths,
        lng         = ~lon,
        lat         = ~lat,
        radius      = 3.5,
        color       = "#1a1a1a",
        weight      = 0.5,
        fillColor   = ~col,
        fillOpacity = 0.35,
        group       = "Moving trail",
        label       = ~paste(bird_id, "-", format(date, "%d %b %Y")),
        options     = pathOptions(pane = "paneTrail", interactive = FALSE))
    })

    # ---- Current-day "latest data points" highlight (ALWAYS ON TOP) --------
    observe({
      cur <- current_frame_data()
      proxy <- leafletProxy(ns("anim_map")) %>%
        clearGroup("Current day")
      if (is.null(cur) || nrow(cur) == 0) return()

      pal <- attr(cur, "palette_colors")
      cur_col <- unname(pal[cur$group_col])
      cur_col[is.na(cur_col)] <- "#444444"

      proxy %>% addCircleMarkers(
        data        = cur,
        lng         = ~lon,
        lat         = ~lat,
        layerId     = ~paste("cur", bird_id, year, sep = "##"),
        radius      = 6,
        color       = "#1a1a1a",
        weight      = 0.9,
        fillColor   = cur_col,
        fillOpacity = 0.70,           # slightly translucent
        group       = "Current day",
        label       = ~paste(bird_id, "-", country, "-",
                             format(date, "%d %b %Y")),
        options     = pathOptions(pane = "paneCurrent"))
    })

    # ---- Selected-bird highlight (click toggles selection) -----------------
    observeEvent(input$anim_map_marker_click, {
      m <- input$anim_map_marker_click
      if (is.null(m) || is.null(m$id)) return()
      lid <- as.character(m$id)
      parts <- strsplit(lid, "##", fixed = TRUE)[[1]]
      # Layout: "pr"  -> ["pr",  bird_id, year, doy]
      #         "cur" -> ["cur", bird_id, year]
      #         "sel" -> ["sel", bird_id, year, yyyymmdd]
      bird <- if (length(parts) >= 2) parts[2] else lid
      # Toggle: second click on the same bird clears the selection.
      if (!is.null(sel_bird()) && identical(sel_bird(), bird)) {
        sel_bird(NULL)
      } else {
        sel_bird(bird)
      }
    })

    observe({
      proxy <- leafletProxy(ns("anim_map")) %>%
        clearGroup("Selected bird")
      bird <- sel_bird()
      if (is.null(bird)) return()
      df <- filtered$daily()
      if (is.null(df) || nrow(df) == 0) return()
      track <- df[df$bird_id == bird, , drop = FALSE]
      if (nrow(track) == 0) return()

      track <- track[order(track$year, track$date), ]
      groups <- split(track, paste(track$bird_id, track$year, sep = "|"))
      groups <- groups[vapply(groups, nrow, integer(1)) >= 2L]
      if (length(groups) > 0) {
        geoms <- lapply(groups, function(g)
          sf::st_linestring(cbind(as.numeric(g$lon), as.numeric(g$lat))))
        sel_sf <- sf::st_sf(bird_id = bird,
                            geometry = sf::st_sfc(geoms, crs = 4326))
        proxy %>% addPolylines(data = sel_sf,
                               color = "#111111",
                               weight = 2.4,
                               opacity = 0.85,
                               group  = "Selected bird",
                               options = pathOptions(pane = "paneSelected"))
      }
      proxy %>% addCircleMarkers(
        data        = track,
        lng         = ~lon,
        lat         = ~lat,
        # layerId encodes the same bird so a second click on any yellow
        # waypoint triggers the toggle handler and deselects.
        layerId     = ~paste("sel", bird_id, year,
                             format(date, "%Y%m%d"), sep = "##"),
        radius      = 4,
        color       = "#111111",
        weight      = 1.2,
        fillColor   = "#FFE45E",
        fillOpacity = 0.9,
        group       = "Selected bird",
        label       = ~paste(bird_id, "-", format(date, "%d %b %Y")),
        options     = pathOptions(pane = "paneSelected"))
    })

    # ---- Selected bird details card ---------------------------------------
    output$bird_card <- renderUI({
      bird <- sel_bird()
      if (is.null(bird)) {
        return(div(class = "small text-muted",
                   "Click any dot on the map to highlight that bird's ",
                   "full track and see its details here. Click the same ",
                   "bird again to clear the selection."))
      }
      df <- filtered$daily()
      track <- df[df$bird_id == bird, , drop = FALSE]
      if (nrow(track) == 0) {
        return(div(class = "small text-muted",
                   sprintf("No data for bird %s in current filter.", bird)))
      }
      years <- sort(unique(track$year))
      tagList(
        tags$div(class = "bird-card",
                 tags$div(tags$b(bird)),
                 tags$div(class = "small",
                          sprintf("%s - %s (%s flyway)",
                                  track$colony_name[1], track$country[1],
                                  track$flyway[1])),
                 tags$div(class = "small text-muted",
                          sprintf("%d fixes across %d year(s): %s",
                                  nrow(track), length(years),
                                  paste(years, collapse = ", "))),
                 tags$div(class = "small text-muted",
                          sprintf("Latitude range: %.1f - %.1f deg",
                                  min(track$lat, na.rm = TRUE),
                                  max(track$lat, na.rm = TRUE))),
                 tags$div(class = "small text-muted",
                          sprintf("Phases observed: %s",
                                  paste(sort(unique(track$phase)),
                                        collapse = ", "))),
                 actionLink(ns("clear_sel"), "Clear selection",
                            class = "small mt-1")
        )
      )
    })
    observeEvent(input$clear_sel, { sel_bird(NULL) })

    # ---- Latitude-of-active-individuals plot (single chart) ----------------
    output$anim_lat_curve <- renderPlotly({
      cur <- current_frame_data()
      shiny::validate(need(!is.null(cur) && nrow(cur) > 0,
                           "No active individuals on this day."))
      pal_cols <- attr(cur, "palette_colors")
      p <- ggplot(cur, aes(x = group_col, y = lat, fill = group_col)) +
        geom_boxplot(alpha = 0.7, outlier.shape = NA) +
        geom_jitter(width = 0.2, size = 1.5, alpha = 0.7) +
        scale_fill_manual(values = pal_cols) +
        scale_y_continuous(limits = c(-15, 60)) +
        labs(x = NULL, y = "Latitude (degrees N)") +
        theme_minimal(base_size = 11) +
        theme(legend.position = "none",
              panel.grid.minor = element_blank())
      ggplotly(p) %>% plotly::config(displayModeBar = TRUE)
    })

    observeEvent(is_playing(), {
      updateActionButton(session, "playBtn",
                         label = if (is_playing()) "Pause" else "Play")
    })
  })
}


## mod_filters

In [6]:
# =============================================================================
# R/mod_filters.R - Filter sidebar module
# =============================================================================
suppressPackageStartupMessages({
  library(shiny)
  library(shinyWidgets)
  library(readr)
  library(dplyr)
})

.colonies_static <- tryCatch(
  readr::read_csv(file.path("data", "raw", "colonies.csv"), show_col_types = FALSE) |> dplyr::arrange(country, colony_name),
  error = function(e) data.frame(colony_name = character(0), country = character(0), flyway = character(0))
)

.choices_grouped <- if (nrow(.colonies_static) > 0) split(.colonies_static$colony_name, .colonies_static$country) else list()
.all_colonies     = unname(unlist(.choices_grouped))
.western_colonies <- .colonies_static$colony_name[.colonies_static$flyway == "western"]
.eastern_colonies <- .colonies_static$colony_name[.colonies_static$flyway == "eastern"]

filters_ui <- function(id) {
  ns <- NS(id)
  tagList(
    h5("Filters", class = "filter-heading"),
    div(class = "quick-filters",
        actionButton(ns("qf_all"), "All", class = "qf-btn"),
        actionButton(ns("qf_western"), "W flyway", class = "qf-btn"),
        actionButton(ns("qf_eastern"), "E flyway", class = "qf-btn"),
        actionButton(ns("qf_clear"), "Clear", class = "qf-btn")
    ),
    pickerInput(ns("country"), label = NULL, choices = .choices_grouped, selected = .all_colonies, multiple = TRUE, options = pickerOptions(actionsBox = TRUE, liveSearch = TRUE, size = 11, selectedTextFormat = "count > 3", countSelectedText = "{0} colonies selected")),
    sliderTextInput(ns("year"), "Year",
                    choices = c("2014", "2015", "2016", "2017"),
                    selected = c("2014", "2017"), grid = TRUE),
    radioButtons(ns("group_mode"), "Color by", choices = c("Country" = "country", "Colony" = "colony", "Flyway" = "flyway", "Year" = "year"), selected = "country", inline = TRUE),
    p(class = "small text-muted mt-2", "Tip: use the W/E flyway buttons to compare populations.")
  )
}

filters_server <- function(id, processed) {
  moduleServer(id, function(input, output, session) {
    observeEvent(input$qf_all, { updatePickerInput(session, "country", selected = .all_colonies) })
    observeEvent(input$qf_western, { updatePickerInput(session, "country", selected = .western_colonies) })
    observeEvent(input$qf_eastern, { updatePickerInput(session, "country", selected = .eastern_colonies) })
    observeEvent(input$qf_clear, { updatePickerInput(session, "country", selected = character(0)) })

    sel_years <- reactive({
      yrs <- as.integer(input$year)
      if (length(yrs) == 2) seq(min(yrs), max(yrs)) else yrs
    })

    scope_data <- reactive({
      req(processed$daily)
      df <- processed$daily
      sel <- input$country
      if (is.null(sel) || length(sel) == 0) return(df[0, , drop = FALSE])
      df |> dplyr::filter(colony_name %in% sel, year %in% sel_years())
    })

    # RESTORED: The phenology reactive filter
    filtered_phenology <- reactive({
      req(processed$phenology)
      sel <- input$country
      if (is.null(sel) || length(sel) == 0) {
        return(processed$phenology[0, , drop = FALSE])
      }
      keep_ids <- processed$daily |>
        dplyr::filter(colony_name %in% sel) |>
        dplyr::pull(colony_id) |> unique()
      processed$phenology |>
        dplyr::filter(colony_id %in% keep_ids, year %in% sel_years())
    })

    list(
      daily      = scope_data,
      phenology  = filtered_phenology, # <--- Added back here!
      years      = sel_years,
      group_mode = reactive(input$group_mode)
    )
  })
}

## mod_metrics

In [7]:
# =============================================================================
# R/mod_metrics.R - Summary KPI strip (lives in the navbar, top-right)
# =============================================================================
suppressPackageStartupMessages({
  library(shiny)
  library(bslib)
})

metrics_ui <- function(id) {
  ns <- NS(id)
  div(class = "navbar-metrics",
      div(class = "navbar-metric",
          span(class = "navbar-metric-value",
               textOutput(ns("n_birds"), inline = TRUE)),
          span(class = "navbar-metric-label", "Individuals tracked")),
      div(class = "navbar-metric",
          span(class = "navbar-metric-value",
               textOutput(ns("n_fixes"), inline = TRUE)),
          span(class = "navbar-metric-label", "Daily fixes")),
      div(class = "navbar-metric",
          span(class = "navbar-metric-value",
               textOutput(ns("n_pops"), inline = TRUE)),
          span(class = "navbar-metric-label", "Populations / colonies"))
  )
}

metrics_server <- function(id, filtered) {
  moduleServer(id, function(input, output, session) {
    output$n_birds <- renderText({
      format(length(unique(filtered$daily()$bird_id)), big.mark = " ")
    })
    output$n_fixes <- renderText({
      format(nrow(filtered$daily()), big.mark = " ")
    })
    output$n_pops <- renderText({
      sprintf("%d / %d",
              length(unique(filtered$daily()$country)),
              length(unique(filtered$daily()$colony_id)))
    })
  })
}


## mod_phenology

In [8]:
# =============================================================================
# R/mod_phenology.R - Phenology / latitude-over-time plot
# -----------------------------------------------------------------------------
# Full-width latitude over the annual cycle.  Each thin coloured line is one
# bird-year track; steep descents = autumn migration, ascents = spring return.
# (Figure 3 / breeding-stay panel has been removed at user request.)
# =============================================================================

suppressPackageStartupMessages({
  library(shiny)
  library(plotly)
  library(dplyr)
  library(ggplot2)
})

phenology_ui <- function(id) {
  ns <- NS(id)
  tagList(
    h6("Latitude across the annual cycle", class = "panel-subheading"),
    plotlyOutput(ns("lat_doy"), height = "360px"),
    div(class = "plot-caption",
        "Latitude (degrees N) across the annual cycle. Each thin ",
        "line is one bird-year. Steep descents = autumn migration; ",
        "ascents = spring return. Lines are split where the tracker ",
        "had no fix for more than 14 days, so no fake interpolation ",
        "is drawn across gaps.")
  )
}

phenology_server <- function(id, filtered, processed) {
  moduleServer(id, function(input, output, session) {

    output$lat_doy <- renderPlotly({ suppressWarnings({
      df <- filtered$daily()
      shiny::validate(need(nrow(df) > 0, "No data for current filter selection."))

      gm <- filtered$group_mode()
      pal <- resolve_palette(df, gm)
      df$group_col <- switch(gm,
                             country = df$country,
                             colony  = df$colony_name,
                             flyway  = df$flyway,
                             year    = as.character(df$year))

      df <- df[order(df$bird_id, df$year, df$date), ]
      df <- df %>%
        dplyr::group_by(bird_id, year) %>%
        dplyr::mutate(
          days_diff   = as.numeric(difftime(date, dplyr::lag(date), units = "days")),
          new_segment = ifelse(is.na(days_diff) | days_diff > 14, 1, 0),
          segment_id  = cumsum(new_segment)
        ) %>%
        dplyr::ungroup() %>%
        dplyr::mutate(line_group = paste(bird_id, year, segment_id))

      p <- ggplot(df, aes(x = doy, y = lat, color = group_col,
                          group = line_group,
                          text = sprintf("%s | %s\n%s",
                                         bird_id, group_col,
                                         format(date, "%d %b %Y")))) +
        geom_line(alpha = 0.22, linewidth = 0.5) +
        scale_color_manual(values = pal$colors, name = NULL) +
        scale_x_continuous(
          breaks = c(1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335),
          labels = c("Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"),
          expand = expansion(mult = c(0.005, 0.005))) +
        labs(x = NULL, y = "Latitude (degrees N)") +
        theme_minimal(base_size = 11) +
        theme(legend.position = "bottom",
              panel.grid.minor = element_blank())

      ggplotly(p, tooltip = "text") %>%
        plotly::config(displayModeBar = TRUE, scrollZoom = TRUE) %>%
        plotly::layout(legend = list(orientation = "h", y = -0.18,
                                     x = 0.5, xanchor = "center"))
    }) })
  })
}


In [1]:
# Detach geosphere to avoid span() conflict
if ("package:geosphere" %in% search()) {
  detach("package:geosphere", unload = TRUE)
}

ui <- fluidPage(
  bslib::page_sidebar(
    sidebar = bslib::sidebar(
      filters_ui("filters"),
      metrics_ui("metrics")
    ),
    animation_ui("animation"),
    phenology_ui("phenology")
  )
)

server <- function(input, output, session) {
  processed <- load_processed()
  filtered  <- filters_server("filters", processed)
  metrics_server("metrics", filtered)
  animation_server("animation", filtered, processed)
  phenology_server("phenology", filtered, processed)
}

shiny::runApp(list(ui = ui, server = server), launch.browser = TRUE)

: [1m[33mError[39m:[22m
[33m![39m could not find function "fluidPage"